# Reproducible MLP: learning XOR from synthetic data

A clean, CPU-only neural-network lesson with **NumPy**, fixed seeds, and independent train/validation/test splits. This is a **new educational reference example**, not a claim that the archived 2020 notebooks have been re-run.

**Objectives:** understand arrays, nonlinear hidden layers, binary cross-entropy, basic gradient descent, train-only normalization, leakage prevention and holdout evaluation.

**Requirements:** Python 3.10+, `numpy>=1.24`. No GPU, network, API key, proprietary dataset or account needed.

**Research caution:** A high score on an artificial XOR dataset is not evidence of broad generalization. We deliberately select no hyperparameters on the test set.

In [ ]:
import numpy as np

SEED = 42
rng = np.random.default_rng(SEED)
X = rng.uniform(-1.0, 1.0, size=(1200, 2))
y = (X[:, 0] * X[:, 1] > 0.0).astype(float).reshape(-1, 1)

order = rng.permutation(len(y))
train_idx, val_idx, test_idx = order[:840], order[840:1020], order[1020:]
X_train, y_train = X[train_idx], y[train_idx]
X_val, y_val = X[val_idx], y[val_idx]
X_test, y_test = X[test_idx], y[test_idx]

assert len(set(train_idx) & set(val_idx)) == 0
assert len(set(train_idx) & set(test_idx)) == 0
assert len(set(val_idx) & set(test_idx)) == 0
print("Samples:", len(train_idx), len(val_idx), len(test_idx))

## Feature scaling without leakage

Compute mean and standard deviation only from the training split; reuse those values unchanged for validation and test.

In [ ]:
mean = X_train.mean(axis=0)
std = X_train.std(axis=0)
assert np.all(std > 0)

A = (X_train - mean) / std
B = (X_val - mean) / std
C = (X_test - mean) / std
print("Train normalization:", np.round(A.mean(axis=0), 6))

## Train a two-layer NumPy MLP

The hidden layer uses `tanh`, and the output is a sigmoid probability. We compute stable binary cross-entropy with `np.logaddexp` and update weights using manually derived gradients.

In [ ]:
hidden = 16
W1 = rng.normal(0.0, 0.35, size=(2, hidden))
b1 = np.zeros((1, hidden))
W2 = rng.normal(0.0, 0.35, size=(hidden, 1))
b2 = np.zeros((1, 1))

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -40, 40)))

def predict_proba(features):
    return sigmoid(np.tanh(features @ W1 + b1) @ W2 + b2)

def accuracy(features, targets):
    return float(np.mean((predict_proba(features) >= 0.5) == targets))

history = []
for epoch in range(1500):
    activations = np.tanh(A @ W1 + b1)
    logits = activations @ W2 + b2
    probabilities = sigmoid(logits)
    loss = float(np.mean(np.logaddexp(0.0, logits) - y_train * logits))
    dlogits = (probabilities - y_train) / len(y_train)
    grad_w2 = activations.T @ dlogits
    grad_b2 = dlogits.sum(axis=0, keepdims=True)
    grad_hidden = (dlogits @ W2.T) * (1.0 - activations**2)
    grad_w1 = A.T @ grad_hidden
    grad_b1 = grad_hidden.sum(axis=0, keepdims=True)

    learning_rate = 0.15
    W1 -= learning_rate * grad_w1
    b1 -= learning_rate * grad_b1
    W2 -= learning_rate * grad_w2
    b2 -= learning_rate * grad_b2

    if epoch in (0, 99, 299, 699, 1499):
        history.append((epoch + 1, round(loss, 4), round(accuracy(B, y_val), 4)))

print("Epoch | Train BCE | Validation accuracy")
for step in history:
    print(step)

## Final untouched holdout and a baseline

This **test set is inspected only after training**. Accuracy is compared with a majority-class baseline. You can rerun the notebook with a new seed and compare variation; no single lucky split should become a performance guarantee.

In [ ]:
train_acc = accuracy(A, y_train)
val_acc = accuracy(B, y_val)
test_acc = accuracy(C, y_test)
majority = float(max(y_test.mean(), 1.0 - y_test.mean()))
print(f"Train: {train_acc:.3f} | Validation: {val_acc:.3f} | Test: {test_acc:.3f}")
print(f"Test majority baseline: {majority:.3f}")
assert len(history) == 5
assert np.isfinite([train_acc, val_acc, test_acc]).all()
assert test_acc > 0.90, "Expected simple synthetic task to be learned."
assert test_acc > majority + 0.20, "MLP should outperform a simple majority baseline."

## Exercises and limitations

1. Replace `tanh` with ReLU and derive the updated backpropagation.
2. Run seeds 1–10 and report average plus variability, not only the best run.
3. Introduce 15% training-label noise and compare training/validation behavior.
4. Add confusion-matrix counts and investigate errors near the decision boundaries.
5. Reproduce the baseline in PyTorch while retaining the same split.

**What this notebook does NOT establish:** state-of-the-art ML, an independent dataset benchmark, a safety-certified classifier, or successful reproduction of all 65 legacy notebooks.

**License:** Repository license applies if one is declared. Otherwise, obtain permission for redistribution.